# Memory API – Quick Tests

Use this notebook to sanity‑check the Memories endpoints locally.

Auth options:
- If server runs with `DISABLE_AUTH_FOR_TESTING=true`, set `TOKEN = 'mock-dev-token'`.
- Otherwise, paste a valid Clerk JWT into `TOKEN`.

Endpoints covered:
- Create (from message_id and raw content)
- List
- Update
- Search
- Stats
- Delete

Notes:
- CID is a companion_id (not a conversation_id).
- UID is optional: the expected owner_id for CID, used only for local checks; the API authorizes using your token.


In [ ]:
# If needed, install dependencies
import json
import subprocess
import sys

try:
    import requests  # type: ignore
except Exception:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "requests"])
    import requests

from typing import Any, Dict

API = "http://localhost:8100"  # base URL
CID = "9ae93c0d-0639-40c7-a9d6-6f8108c2f131"
UID = ""  # optional: expected owner ID (for sanity checks only)
UID = "e3947eb9-b9a7-417d-a853-02077494592f"
MESSAGE_UUID = "1b1605e2-d2a3-4370-a632-bbaeb4871581"
TOKEN = "mock-dev-token"  # or paste a real Clerk JWT

# Try to fetch a dev token from the API (works if DISABLE_AUTH_FOR_TESTING=true)
try:
    r = requests.get(f"{API}/api/auth/dev-token", timeout=5)
    if r.ok:
        token_data = r.json()
        if isinstance(token_data, dict) and token_data.get("token"):
            TOKEN = token_data["token"]
            print("Using dev token from API")
except Exception as _e:
    pass


def api(
    path: str,
    method: str = "GET",
    json_body: Dict[str, Any] | None = None,
    params: Dict[str, Any] | None = None,
):
    url = f"{API}{path}"
    headers = {"Content-Type": "application/json"}
    if TOKEN:
        headers["Authorization"] = f"Bearer {TOKEN}"
    resp = requests.request(
        method.upper(), url, headers=headers, json=json_body, params=params, timeout=60
    )
    try:
        data = resp.json()
    except Exception:
        data = resp.text
    print(f"{method.upper()} {path} -> {resp.status_code}")
    if isinstance(data, (dict, list)):
        print(json.dumps(data, indent=2, default=str)[:2000])
    else:
        print(str(data)[:2000])
    resp.raise_for_status()
    return data


# Inspect current auth user and verify ownership
CURRENT_UID = None
try:
    me = api("/api/me")
    CURRENT_UID = me.get("id")
    print("Authenticated user id:", CURRENT_UID)
    if UID and UID != CURRENT_UID:
        print(
            "Warning: Provided UID does not match authenticated user; ownership checks will fail if CID belongs to UID."
        )
except Exception as _e:
    print("Could not fetch /api/me; proceeding without user check.")

# Verify companion ownership; if not owned by current user, try to auto-select one
try:
    _ = api(f"/api/companions/{CID}")
    print("Companion ownership verified for current user.")
except Exception as e:
    print("Provided CID not accessible to current user:", e)
    try:
        comp = api("/api/companion-name")
        if isinstance(comp, dict) and comp.get("id"):
            CID = comp["id"]
            print("Using companion for current user:", CID)
    except Exception as _e2:
        print("Could not auto-resolve a companion for the current user.")

In [ ]:
# 1) Create memory from an existing message (unmasked)
created_ids = []
try:
    r = api(
        f"/api/companions/{CID}/memories",
        "POST",
        json_body={"message_id": MESSAGE_UUID, "sender_type": "user"},
    )
    if isinstance(r, dict) and "id" in r:
        created_ids.append(r["id"])
except Exception as e:
    print("Create-from-message failed:", e)

# 2) Create memory with raw content
try:
    r2 = api(
        f"/api/companions/{CID}/memories",
        "POST",
        json_body={"content": "User likes hiking", "sender_type": "user"},
    )
    if isinstance(r2, dict) and "id" in r2:
        created_ids.append(r2["id"])
except Exception as e:
    print("Create-with-content failed:", e)

In [ ]:
# 3) List recent memories
_ = api(
    f"/api/companions/{CID}/memories",
    "GET",
    params={"limit": 10, "order_by": "created_at", "order_dir": "DESC"},
)

In [ ]:
# 4) Update the last created memory (if any)
if created_ids:
    mid = created_ids[-1]
    _ = api(
        f"/api/memories/{mid}",
        "PUT",
        json_body={"importance": 0.9, "commentary": "test update via notebook"},
    )
else:
    print("No created memory to update")

In [ ]:
# 5) Search
_ = api(
    f"/api/companions/{CID}/memories/search",
    "POST",
    json_body={"query": "What activities does the user enjoy?", "top_k": 10, "min_saliency": 0.2},
)

In [ ]:
# 6) Stats
_ = api(f"/api/companions/{CID}/memories/stats")

In [ ]:
# 7) Cleanup: delete any memory created above
for mid in created_ids:
    try:
        _ = api(f"/api/memories/{mid}", "DELETE")
    except Exception as e:
        print("Delete failed for", mid, e)